In [2]:
import pandas as pd
import xgboost as xgb
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score

In [3]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [6]:
# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 900].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 99
Number of rows left: 101004


In [7]:
# Instead, calculate class weights inversely proportional to class frequencies
class_counts = pd.Series(y_train).value_counts()
total_samples = len(y_train)
class_weights = {class_idx: total_samples / (len(class_counts) * count) 
                for class_idx, count in class_counts.items()}

In [8]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(np.unique(y_train)),
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        
        # Try including early stopping directly in the model parameters
        'early_stopping_rounds': 50,
    }
    
    # Create XGBoost model with parameters from Optuna
    model = xgb.XGBClassifier(**params)
    
     # Create sample_weight array based on class weights
    sample_weight = np.array([class_weights[y] for y in y_train])
    
    model.fit(
        X_train,  # Use original training data instead of resampled
        y_train,  # Use original labels instead of resampled
        eval_set=[(X_test, y_test)],
        sample_weight=sample_weight,  # Add sample weights
        verbose=False
    )
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

In [9]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="XGboost_diseases_symptoms_dropextremelymore900withoutSMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/xgboost.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-23 12:32:15,747] A new study created in RDB with name: XGboost_diseases_symptoms_dropextremelymore900withoutSMOTE_study
[I 2025-04-23 12:34:27,014] Trial 0 finished with value: 0.496757586258106 and parameters: {'max_depth': 15, 'learning_rate': 0.24523086213324977, 'n_estimators': 580, 'subsample': 0.9042272655435726, 'colsample_bytree': 0.547408376854357, 'gamma': 4.086905304914365, 'reg_alpha': 1.0219971405651296, 'reg_lambda': 3.383565867599197}. Best is trial 0 with value: 0.496757586258106.
[I 2025-04-23 12:37:21,614] Trial 1 finished with value: 0.4993317162516707 and parameters: {'max_depth': 14, 'learning_rate': 0.29957132094173095, 'n_estimators': 806, 'subsample': 0.560628977228038, 'colsample_bytree': 0.8202297875459776, 'gamma': 4.2646718303271545, 'reg_alpha': 1.121610770734836, 'reg_lambda': 2.5330372798908347}. Best is trial 1 with value: 0.4993317162516707.
[I 2025-04-23 12:38:50,353] Trial 2 finished with value: 0.49269838126825405 and parameters: {'max_dep


Best Trial:
FrozenTrial(number=19, state=TrialState.COMPLETE, values=[0.5013613187465967], datetime_start=datetime.datetime(2025, 4, 23, 13, 14, 3, 267033), datetime_complete=datetime.datetime(2025, 4, 23, 13, 15, 10, 836024), params={'max_depth': 12, 'learning_rate': 0.09640940141776703, 'n_estimators': 669, 'subsample': 0.7819573071602194, 'colsample_bytree': 0.8569263691294355, 'gamma': 0.4956027063282902, 'reg_alpha': 4.190339062892318, 'reg_lambda': 2.418983862769093}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=15, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'reg_alpha':